# databricks-external-access: tutorial

Browse a Databricks workspace and fetch a table to your machine as Parquet, without
Databricks compute.

**Before you start**

1. Run the next cell once per machine with your workspace URL; it is remembered afterwards.
2. `login` opens a browser to sign in through your workspace's SSO. After that the
   token is cached and refreshed automatically.
3. Fetching needs `SELECT` on the table and `EXTERNAL USE SCHEMA` on its schema (ask your
   Databricks admin). Browsing works without them.

In [ ]:
import databricks_external_access as dbx

# Run once per machine, then you can delete or comment out this line:
dbx.login("https://your-workspace-url")  # <- edit (any workspace URL, custom ones included)

## 1. Browse

**The whole picture in one call.** `overview()` prints every catalog, schema and table you can
see as a tree, with table kinds and descriptions. Pass a catalog name to limit it, e.g.
`dbx.overview("your_catalog")`. It skips Databricks' own `system` and `samples` catalogs
unless you name them.

In [ ]:
dbx.overview()

**Or step by step.** The same information, one level at a time, as plain Python objects you can
use in your own code. What can I see?

In [ ]:
for catalog in dbx.catalogs():
    print(f"{catalog.name:20} {catalog.comment or ''}")

Each level feeds the next: pick a catalog from the list above, then list its **schemas**
(`schemas("catalog")` returns names like `catalog.schema`, which you can pass straight on).

In [ ]:
CATALOG = "your_catalog"  # <- edit: one of the catalogs above

for schema in dbx.schemas(CATALOG):
    print(f"{schema.name:30} {schema.comment or ''}")

Pick a schema from that list and list its **tables**, with their kind and description.

In [ ]:
SCHEMA = f"{CATALOG}.your_schema"  # <- edit: one of the schemas above

for table in dbx.tables(SCHEMA):
    print(f"{table.name:50} {table.kind:18} {table.comment or ''}")

Pick a table and look at its columns.

In [ ]:
TABLE = f"{SCHEMA}.your_table"  # <- edit

info = dbx.describe(TABLE)
print(info.name, "-", info.kind, "-", info.comment)
for column in info.columns:
    print(f"  {column.name:30} {column.type:15} {column.comment or ''}")

## 2. Fetch

`fetch` prints the table's size on storage **before** reading any data. Use `columns`, `where`
(a DuckDB SQL predicate) and `limit` to fetch just a slice; the size shown is always the whole
table, an upper bound on what is read. Start small.

In [ ]:
path = dbx.fetch(TABLE, limit=1000, dest="data")
path

## 3. Use the data

The result is a plain Parquet file: load it with any framework.

In [ ]:
import pandas as pd

df = pd.read_parquet(path)
df.head()

In [ ]:
import duckdb

duckdb.sql(f"SELECT count(*) AS rows FROM '{path.as_posix()}'").show()

## 4. Errors are meant to be read

Every failure raises `databricks_external_access.Error` with a message saying what to do.
Try a table that does not exist:

In [ ]:
try:
    dbx.fetch(f"{SCHEMA}.this_table_does_not_exist", dest="data")
except dbx.Error as error:
    print(error)